In [ ]:
# In the first cell, ensure these imports are present or add them:
import shutil
import os
from pathlib import Path

# Support launching Jupyter from either the repository root or code/.
START_DIR = Path.cwd().resolve()
CODE_DIR = START_DIR if (START_DIR / 'src').is_dir() else START_DIR / 'code'
if not (CODE_DIR / 'src').is_dir():
    raise FileNotFoundError('Could not locate code/src. Start Jupyter from the repository root or code directory.')
REPO_ROOT = CODE_DIR.parent
os.chdir(CODE_DIR)

cache_dirs = [CODE_DIR / 'src/model/__pycache__', CODE_DIR / 'src/util/__pycache__']
for cache_dir in cache_dirs:
    if cache_dir.exists(): shutil.rmtree(cache_dir)

from transformers import BitsAndBytesConfig as TransformersBitsAndBytesConfig

import torch
import numpy as np
from pytorch_lightning import seed_everything
import joblib
from torch.optim import Adam
from PIL import Image as PILImage # Using an alias for clarity, original uses 'Image'
import pandas as pd

import json
import base64
import io
import requests

from src.util.utils import show_image, reset_attn, read_image, prompt_embd_aligned_replacement, save_image, load_pipe_multi
from src.util.prompt_runing_multi import save_inversion_latents, run_baseline, replace_enhance

In [ ]:
device_map = {
    'vae': 'cuda:0',
    'text_encoder': 'cuda:0',
    'transformer': 'cuda:0'
}

pipe = load_pipe_multi(device_map)


In [ ]:

# --- Define Raw Image Path and Edit Instruction ---
raw_image_path = str(CODE_DIR / "example.jpg")
edit_instruction = "let the bear raise its hand"
original_source_prompt =  "A polar bear standing on an ice field."
original_target_prompt =  "A polar bear standing on an ice field raising its hand."
# Basic parameters
HEIGHT = 512 # Desired height for the diffusion pipeline

STEPS = 25
SEED = 2

emphasize_scale = 5

# Prompt enhancement

In [ ]:
# --- Configuration for LLM ---
# IMPORTANT: Set your OpenRouter API key (can be set directly or as an environment variable)
os.environ["OPENROUTER_API_KEY"] = "" #Enter your own OpenRouter API key here

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY environment variable not set. Please set it before running.")

# --- Image Loading and Preparation for LLM ---
def encode_image_to_base64_for_llm(image_path):
    try:
        img = PILImage.open(image_path)
        if img.mode == 'RGBA' or img.mode == 'LA' or (img.mode == 'P' and 'transparency' in img.info):
            img = img.convert('RGB')
        output_buffer = io.BytesIO()
        img.save(output_buffer, format="JPEG")
        byte_data = output_buffer.getvalue()
        return base64.b64encode(byte_data).decode('utf-8')
    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
        raise
    except Exception as e:
        print(f"Error encoding image {image_path}: {e}")
        raise

print(f"Loading and encoding image for LLM: {raw_image_path}")
base64_image = encode_image_to_base64_for_llm(raw_image_path)

# --- LLM Call to Generate Prompts ---
llm_model_name = "google/gemini-2.5-pro"

# Load system prompt from file
system_prompt_file = str(REPO_ROOT / "mappings" / "system_prompt.txt")
system_prompt_llm = None

# Load the shared prompt template from mappings/.
if os.path.exists(system_prompt_file):
    try:
        with open(system_prompt_file, 'r', encoding='utf-8') as f:
            system_prompt_llm = f.read()
        print(f"Successfully loaded system prompt from {os.path.abspath(system_prompt_file)}")
    except Exception as e:
        print(f"Error reading {system_prompt_file}: {e}")

# Get original prompts if they exist
original_source_prompt = globals().get('original_source_prompt', '')
original_target_prompt = globals().get('original_target_prompt', '')

# If mappings/system_prompt.txt doesn't exist, use the original prompts directly.
if system_prompt_llm is None:
    if original_source_prompt and original_target_prompt:
        print(f"{system_prompt_file} not found. Using original_source_prompt and original_target_prompt directly.")
        generated_source_prompt = original_source_prompt
        generated_target_prompt = original_target_prompt
    else:
        raise FileNotFoundError(f"{system_prompt_file} not found and original_source_prompt/original_target_prompt are not available. Please provide mappings/system_prompt.txt or set original_source_prompt and original_target_prompt.")
else:
    # Replace placeholders in system prompt if they exist
    # Note: system_prompt.txt may contain {EDIT_INSTRUCTION}, {ORIGINAL_SOURCE_PROMPT}, {ORIGINAL_TARGET_PROMPT}
    
    # Check which placeholders exist before replacement
    has_edit_instruction_placeholder = '{EDIT_INSTRUCTION}' in system_prompt_llm
    has_old_format_placeholder = '{}' in system_prompt_llm
    
    # Replace placeholders (handle both {EDIT_INSTRUCTION} and {} format)
    if has_edit_instruction_placeholder:
        system_prompt_llm = system_prompt_llm.replace('{EDIT_INSTRUCTION}', edit_instruction)
    if '{ORIGINAL_SOURCE_PROMPT}' in system_prompt_llm:
        system_prompt_llm = system_prompt_llm.replace('{ORIGINAL_SOURCE_PROMPT}', original_source_prompt)
    if '{ORIGINAL_TARGET_PROMPT}' in system_prompt_llm:
        system_prompt_llm = system_prompt_llm.replace('{ORIGINAL_TARGET_PROMPT}', original_target_prompt)
    # Also handle the old format with {} for edit_instruction (only if {EDIT_INSTRUCTION} was not used)
    if has_old_format_placeholder and not has_edit_instruction_placeholder:
        system_prompt_llm = system_prompt_llm.format(edit_instruction)
    
    # Constructing the messages payload for OpenRouter
    # Note: The structure for multimodal input can vary slightly between models.
    # This structure is common for Claude 3 and Gemini Vision. GPT-4o also supports this.
    # Note: system_prompt_llm already has placeholders replaced above
    messages_payload = [
        {"role": "system", "content": system_prompt_llm},
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"Process the following image according to the edit instruction: \"{edit_instruction}\". Generate the enhanced source and target prompts as per the system instructions."
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}",
                    }
                }
            ]
        }
    ]
    
    print(f"Sending request to OpenRouter API (model: {llm_model_name}) for prompt generation...")
    # Initialize fallback prompts (used only if LLM call fails)
    generated_source_prompt = f"Fallback: Original image at {os.path.basename(raw_image_path)}."
    generated_target_prompt = f"Fallback: Image at {os.path.basename(raw_image_path)} with edit: {edit_instruction}."
    
    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                "Content-Type": "application/json" # Important to specify content type for JSON payload
            },
            data=json.dumps({
                "model": llm_model_name,
                "messages": messages_payload,
                "max_tokens": 5000,
                "temperature": 0, # Lower for more deterministic output
            })
        )
        response.raise_for_status() # Will raise an HTTPError if the HTTP request returned an unsuccessful status code

        completion_data = response.json()
        
        if not completion_data.get("choices") or not completion_data["choices"][0].get("message") or not completion_data["choices"][0]["message"].get("content"):
            print("LLM response structure is not as expected. Full response:")
            print(json.dumps(completion_data, indent=2))
            raise ValueError("LLM response missing expected content path.")

        llm_output_raw = completion_data["choices"][0]["message"]["content"].strip()
        
        print("--- LLM Raw Output ---")
        print(llm_output_raw)
        print("----------------------")

        json_str_cleaned = llm_output_raw
        if json_str_cleaned.startswith("```json"):
            json_str_cleaned = json_str_cleaned[7:]
            if json_str_cleaned.endswith("```"):
                json_str_cleaned = json_str_cleaned[:-3]
        elif not (json_str_cleaned.startswith("{") and json_str_cleaned.endswith("}")):
            json_start = json_str_cleaned.find('{')
            json_end = json_str_cleaned.rfind('}') + 1
            if json_start != -1 and json_end > json_start:
                json_str_cleaned = json_str_cleaned[json_start:json_end]
            else:
                raise ValueError("No clear JSON object found in LLM output. Check raw output.")
        
        llm_output_json = json.loads(json_str_cleaned.strip())
                
        temp_source_prompt = llm_output_json.get("enhanced_source_prompt")
        temp_target_prompt = llm_output_json.get("enhanced_target_prompt")

        if not temp_source_prompt or not temp_target_prompt:
            raise ValueError("LLM output JSON does not contain 'enhanced_source_prompt' or 'enhanced_target_prompt', or they are empty.")
        
        generated_source_prompt = temp_source_prompt
        generated_target_prompt = temp_target_prompt
        print(f"\nSuccessfully parsed prompts from LLM.")

    except requests.exceptions.HTTPError as e:
        print(f"OpenRouter API HTTP Error: {e}")
        print(f"Status Code: {e.response.status_code}, Response Text: {e.response.text}")
        print("Using fallback prompts.")
    except requests.exceptions.RequestException as e:
        print(f"OpenRouter API Request Error: {e}")
        print("Using fallback prompts.")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from LLM output: {e}")
        print("LLM output might not be valid JSON. Please check the raw output above. Using fallback prompts.")
    except ValueError as e:
        print(f"Error processing LLM output: {e}. Using fallback prompts.")
    except Exception as e:
        print(f"An unexpected error occurred during LLM communication: {e}. Using fallback prompts.")

# These global variables will be used by subsequent cells
source_prompt = generated_source_prompt
target_prompt = generated_target_prompt
image_path = raw_image_path.split('/')[-1].split('.')[0]

print(f"\n--- Prompts for Image Editing Pipeline ---")
print(f"Source Prompt: {source_prompt}")
print(f"Target Prompt: {target_prompt}")
print(f"Image for processing: {raw_image_path}")
print(f"Edit instruction given to LLM: {edit_instruction}")


## Info

In [ ]:

# 'source_prompt', 'target_prompt', and 'raw_image_path' should be defined by the LLM cell
# Add fallbacks in case the LLM cell was skipped or failed critically.
if 'raw_image_path' not in globals() or not os.path.exists(raw_image_path):
    print("Warning: 'raw_image_path' from LLM cell is not valid. Using a default image path.")
    raw_image_path = 'dataset/object_addition/1.jpg' # Default if LLM cell fails/skipped
    # If raw_image_path is not set, source/target prompts would also be fallbacks.
if 'source_prompt' not in globals() or "Fallback" in source_prompt:
    print("Warning: 'source_prompt' from LLM cell is not valid or is a fallback. Using a default source prompt.")
    source_prompt = "A deserted highway near snow mountains under a partly cloudy sky" # Default
if 'target_prompt' not in globals() or "Fallback" in target_prompt:
    print("Warning: 'target_prompt' from LLM cell is not valid or is a fallback. Using a default target prompt.")
    target_prompt = "A red car on a deserted highway near snow mountains under a partly cloudy sky" # Default
if 'edit_instruction' not in globals(): # edit_instruction is defined in the LLM cell
    edit_instruction = "Default edit: add red car"


print(f"\nLoading image for pipeline processing: {raw_image_path}")
image = read_image(raw_image_path, HEIGHT) # read_image resizes to HEIGHT, adjusts width
WIDTH = image.shape[2] # Get width from the loaded and (potentially) resized image

print(f"Pipeline Dimensions: HEIGHT={HEIGHT}, WIDTH={WIDTH}")
print(f"Using Source Prompt: {source_prompt}")
print(f"Using Target Prompt: {target_prompt}")

# Display the image that will be used in the pipeline
show_image(image) # show_image is from your utils

# Prepare timesteps for the diffusion process
timesteps, steps = pipe.prepare_timesteps(STEPS, (WIDTH // 16) * (HEIGHT // 16), device_map['vae'])
print(f"Timesteps prepared. Number of inference steps: {steps}")

In [ ]:
# prepare configurations for editing
latents_base_dir = "latents"
latents_save_dir = os.path.join(latents_base_dir,source_prompt[:50])

base_config = {
    "device_map": device_map,
    "steps": STEPS,
    "first_order": False,
    "width": WIDTH,
    "height": HEIGHT,
    "seed": SEED
}

edit_config = {
    "latents_save_dir": latents_save_dir,
    "source_prompt": source_prompt,
    "target_prompt": target_prompt,
    "guidance_scale": 2,
}
base_dir = os.path.join("edit_images")
save_dir = os.path.join(base_dir,"example")
os.makedirs(base_dir,exist_ok=True)
os.makedirs(save_dir,exist_ok=True)
save_image(image, os.path.join(save_dir, f"{image_path}.jpg"))

# Inversion

In [ ]:
# Do inversion and reconstruction
source_x0, source_xT, source_img = save_inversion_latents(pipe,source_prompt,timesteps,latents_save_dir,image,**base_config)
reconstruction = run_baseline(pipe, source_prompt, start_latents=source_xT, guidance_scale=1, **base_config)
target = run_baseline(pipe, target_prompt, start_latents=source_xT, guidance_scale=2, **base_config, emphasize_scale=5)
save_image(image,os.path.join(base_dir,'source.png'))
save_image(reconstruction["images"][0],os.path.join(base_dir,'reconstruction.png'))
save_image(target["images"][0],os.path.join(base_dir,'target.png'))
show_image(image)
show_image(reconstruction["images"][0])
show_image(target["images"][0])

# Attention Reinforcement and Manipulation

In [ ]:
# Perform editing
time = timesteps[0:4]
edit = replace_enhance(pipe,
               **base_config,
               **edit_config,
               timesteps=time,
               emphasize_scale=emphasize_scale)
show_image(edit['images'][0])
save_image(edit['images'][0],os.path.join(save_dir,'edit.png'))